# Hybrid Movie Recommendation - Notebook sạch theo đúng quy trình

Notebook này được sắp xếp lại theo đúng thứ tự bạn yêu cầu:

1. Import thư viện  
2. Đọc dữ liệu  
3. Kiểm tra dữ liệu  
4. Tiền xử lý dữ liệu  
5. Trích chọn đặc trưng TF-IDF  
6. Chia train/dev/test  
7. Huấn luyện SVD  
8. Đánh giá SVD bằng RMSE/MAE  
9. Xây dựng Hybrid Recommendation  
10. Đánh giá Top-K  
11. Demo gợi ý  
12. Lưu kết quả và model  

Mục tiêu của bản này là **không để code bị lẫn lộn**, **không viết lặp nhiều hàm giống nhau**, và mỗi phần đều có ghi chú giải thích trước khi chạy.

## 1. Import thư viện

Phần này chỉ dùng để nạp các thư viện cần thiết cho toàn bộ notebook.

- `pandas`, `numpy`: xử lý dữ liệu dạng bảng và tính toán số học.
- `train_test_split`: chia dữ liệu thành train/dev/test.
- `TfidfVectorizer`: biến nội dung phim dạng chữ thành vector số.
- `cosine_similarity`: tính độ giống nhau giữa hồ sơ sở thích user và phim.
- `mean_squared_error`, `mean_absolute_error`: đánh giá lỗi dự đoán rating của SVD.
- `vstack`: ghép các vector TF-IDF của nhiều phim.
- `surprise`: thư viện dùng để huấn luyện SVD cho hệ gợi ý.

In [ ]:
import os
import pickle

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error

from scipy.sparse import vstack

from surprise import Dataset, Reader, SVD


Nếu máy chưa cài `scikit-surprise`, chạy cell dưới một lần. Nếu đã cài rồi thì bỏ qua.

In [ ]:
# !pip install scikit-surprise


## 2. Đọc dữ liệu

Ở phần này, notebook đọc 4 file dữ liệu gốc từ thư mục `data`:

- `ratings.csv`: dữ liệu đánh giá phim của người dùng.
- `movies.csv`: thông tin phim gồm `movieId`, `title`, `genres`.
- `tags.csv`: tag do người dùng gắn cho phim.
- `links.csv`: liên kết tới các nguồn dữ liệu khác.

Trong mô hình chính, nhóm dùng `ratings.csv`, `movies.csv`, `tags.csv`. File `links.csv` được đọc để kiểm tra đầy đủ dữ liệu nhưng không dùng trực tiếp trong mô hình Hybrid.

In [ ]:
ratings_df = pd.read_csv("data/ratings.csv")
movies_df = pd.read_csv("data/movies.csv")
tags_df = pd.read_csv("data/tags.csv")
links_df = pd.read_csv("data/links.csv")

print("Ratings:", ratings_df.shape)
print("Movies:", movies_df.shape)
print("Tags:", tags_df.shape)
print("Links:", links_df.shape)


Kiểm tra nhanh vài dòng đầu của từng bảng dữ liệu.

In [ ]:
ratings_df.head()


In [ ]:
movies_df.head()


In [ ]:
tags_df.head()


## 3. Kiểm tra dữ liệu

Phần này kiểm tra dữ liệu trước khi tiền xử lý:

1. Kiểm tra dữ liệu trùng lặp.
2. Chuẩn bị bảng rating cho SVD.
3. Kiểm tra số lượng user, số lượng phim, số lượt rating.
4. Tính độ thưa của ma trận rating.

Độ thưa cao nghĩa là đa số user chưa rating đa số phim. Đây là lý do cần dùng SVD/Funk SVD để học trên các rating đã có.

In [ ]:
print("Ratings duplicate:", ratings_df.duplicated().sum())
print("Movies duplicate:", movies_df.duplicated().sum())
print("Tags duplicate:", tags_df.duplicated().sum())


Chuẩn bị dữ liệu rating cho SVD. Mô hình SVD chỉ cần 3 cột:

- `userId`
- `movieId`
- `rating`

In [ ]:
ratings_svd = ratings_df[
    ["userId", "movieId", "rating"]
].copy()

ratings_svd.info()
ratings_svd.head()


In [ ]:
ratings_svd["rating"].describe()


In [ ]:
n_users = ratings_svd["userId"].nunique()
n_movies = ratings_svd["movieId"].nunique()
n_ratings = len(ratings_svd)

print("Số user:", n_users)
print("Số phim có rating:", n_movies)
print("Số lượt rating:", n_ratings)

sparsity = (
    1 - n_ratings / (n_users * n_movies)
) * 100

print(f"Sparsity: {sparsity:.2f}%")


## 4. Tiền xử lý dữ liệu

Phần này xử lý dữ liệu phim và tag để tạo cột `content`.

Cột `content` sẽ là văn bản tổng hợp từ:

```text
title + genres + tag
```

Sau đó cột này được dùng cho TF-IDF ở phần tiếp theo.

### 4.1. Tách năm phát hành từ title

Trong `movies.csv`, title thường có dạng:

```text
Toy Story (1995)
```

Ta tách `1995` ra cột `year` để title sạch hơn.

In [ ]:
movies_df["year"] = movies_df["title"].str.extract(r"\((\d{4})\)")

movies_df["year"] = pd.to_numeric(
    movies_df["year"],
    errors="coerce"
)

movies_df.head()


### 4.2. Xóa năm khỏi title

Sau khi đã tách năm, ta xóa phần `(1995)` khỏi title để nội dung phim gọn hơn.

In [ ]:
movies_df["title"] = (
    movies_df["title"]
    .str.replace(r"\(\d{4}\)", "", regex=True)
    .str.strip()
)

movies_df.head()


### 4.3. Chuẩn hóa genres

Một số phim có giá trị `(no genres listed)`, nghĩa là không có thể loại. Ta đổi giá trị này thành chuỗi rỗng.

Dữ liệu gốc dùng dấu `|` để ngăn cách thể loại, ví dụ:

```text
Adventure|Animation|Children
```

Ta đổi dấu `|` thành khoảng trắng để đưa vào TF-IDF.

In [ ]:
count_no_genres = len(
    movies_df[movies_df["genres"] == "(no genres listed)"]
)

print("Số phim không có genres:", count_no_genres)

movies_df["genres"] = movies_df["genres"].replace(
    "(no genres listed)",
    ""
)

movies_df["genres"] = movies_df["genres"].str.replace(
    "|",
    " ",
    regex=False
)

movies_df.head()


### 4.4. Chuẩn hóa tag

Tag là các từ khóa do user gắn cho phim. Ta đưa tag về chữ thường và xóa khoảng trắng thừa.

In [ ]:
tags_df["tag"] = (
    tags_df["tag"]
    .astype(str)
    .str.lower()
    .str.strip()
)

tags_df.head()


### 4.5. Gom tag theo từng phim

Một phim có thể có nhiều tag từ nhiều user. Ta gom tất cả tag của cùng một `movieId` thành một chuỗi.

In [ ]:
movie_tags = (
    tags_df
    .groupby("movieId")["tag"]
    .apply(" ".join)
    .reset_index()
)

movie_tags.head()


### 4.6. Loại bỏ tag trùng lặp

Nếu một từ tag xuất hiện nhiều lần trong cùng một phim, ta giữ lại một lần để giảm nhiễu.

In [ ]:
movie_tags["tag"] = movie_tags["tag"].apply(
    lambda x: " ".join(
        dict.fromkeys(x.split())
    )
)

movie_tags.head()


### 4.7. Ghép tag vào dữ liệu phim

Ta ghép bảng `movie_tags` vào `movies_df` để mỗi phim có thêm cột `tag`.

In [ ]:
movies_nlp = movies_df.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies_nlp["tag"] = movies_nlp["tag"].fillna("")

movies_nlp.head()


### 4.8. Tạo cột content

Cột `content` là dữ liệu văn bản chính dùng cho TF-IDF.

Nó gồm:

```text
title + genres + tag
```

In [ ]:
movies_nlp["content"] = (
    movies_nlp["title"]
    + " "
    + movies_nlp["genres"]
    + " "
    + movies_nlp["tag"]
)

movies_nlp.head()


### 4.9. Làm sạch content

Các bước làm sạch gồm:

1. Chuyển về chữ thường.
2. Kiểm tra ký tự đặc biệt.
3. Loại bỏ ký tự đặc biệt.
4. Chuẩn hóa khoảng trắng.

In [ ]:
movies_nlp["content"] = (
    movies_nlp["content"]
    .str.lower()
)

special_content = movies_nlp[
    movies_nlp["content"].str.contains(
        r"[^a-zA-Z0-9\s]",
        regex=True,
        na=False
    )
]

print("Số dòng content có ký tự đặc biệt:", len(special_content))

movies_nlp["content"] = (
    movies_nlp["content"]
    .str.replace(
        r"[^a-zA-Z0-9\s]",
        " ",
        regex=True
    )
)

movies_nlp["content"] = (
    movies_nlp["content"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

movies_nlp.head()


## 5. Trích chọn đặc trưng TF-IDF

TF-IDF biến nội dung phim dạng chữ thành vector số.

Sau bước này:

- Mỗi dòng tương ứng với một phim.
- Mỗi cột tương ứng với một từ/đặc trưng trong từ vựng.
- Giá trị trong ma trận thể hiện mức độ quan trọng của từ đó với phim.

Tham số `stop_words="english"` giúp loại bỏ các từ tiếng Anh phổ biến như `the`, `a`, `and`, `of`,... vì các từ này ít giúp phân biệt nội dung phim.

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies_nlp["content"]
)

print("Kích thước ma trận TF-IDF:", tfidf_matrix.shape)

feature_names = tfidf.get_feature_names_out()

print("200 từ đầu tiên trong từ vựng:")
print(feature_names[:200])


## 6. Chia train/dev/test

Dữ liệu rating được chia thành 3 tập:

- `train`: dùng để huấn luyện SVD và xây dựng hồ sơ sở thích người dùng.
- `dev`: dùng để thử nhiều giá trị alpha và chọn cấu hình tốt.
- `test`: dùng để đánh giá cuối cùng.

Tỷ lệ chia:

```text
70% train - 15% dev - 15% test
```

In [ ]:
ratings_train, ratings_temp = train_test_split(
    ratings_svd,
    test_size=0.30,
    random_state=42
)

ratings_dev, ratings_test = train_test_split(
    ratings_temp,
    test_size=0.50,
    random_state=42
)

print("--- KÍCH THƯỚC DỮ LIỆU SAU KHI CHIA ---")
print("Train:", ratings_train.shape)
print("Dev:", ratings_dev.shape)
print("Test:", ratings_test.shape)

print("Tỷ lệ Train:", len(ratings_train) / len(ratings_svd))
print("Tỷ lệ Dev:", len(ratings_dev) / len(ratings_svd))
print("Tỷ lệ Test:", len(ratings_test) / len(ratings_svd))


## 7. Huấn luyện SVD

Phần này huấn luyện mô hình SVD trên tập `ratings_train`.

SVD học từ dữ liệu:

```text
userId, movieId, rating
```

Mục tiêu là dự đoán rating mà user có thể dành cho các phim chưa xem.

Các hyper-parameter đang dùng:

- `n_factors=50`: số chiều đặc trưng ẩn.
- `n_epochs=20`: số vòng học.
- `lr_all=0.005`: tốc độ học.
- `reg_all=0.02`: hệ số giảm overfitting.
- `random_state=42`: cố định kết quả khi chạy lại.

In [ ]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

data_surprise = Dataset.load_from_df(
    ratings_train[["userId", "movieId", "rating"]],
    reader
)

trainset = data_surprise.build_full_trainset()

svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("Đã huấn luyện xong mô hình SVD.")


Kiểm tra ma trận đặc trưng ẩn mà SVD đã học được:

- `pu`: ma trận đặc trưng ẩn của user.
- `qi`: ma trận đặc trưng ẩn của phim.

In [ ]:
print("--- KẾT QUẢ TRÍCH CHỌN ĐẶC TRƯNG SVD ---")

user_features = svd_model.pu
movie_features = svd_model.qi

print(f"Kích thước ma trận đặc trưng ẩn Người dùng (P): {user_features.shape}")
print(f"Kích thước ma trận đặc trưng ẩn Bộ phim (Q): {movie_features.shape}")

print("Vector đặc trưng ẩn của User đầu tiên - 10 chiều đầu:")
print(user_features[0][:10])

print("Vector đặc trưng ẩn của Movie đầu tiên - 10 chiều đầu:")
print(movie_features[0][:10])


## 8. Đánh giá SVD bằng RMSE và MAE

Phần này đánh giá khả năng dự đoán rating của SVD.

- `RMSE`: sai số bình phương trung bình căn bậc hai. Chỉ số này phạt mạnh các lỗi lớn.
- `MAE`: sai số tuyệt đối trung bình. Chỉ số này dễ hiểu hơn, cho biết trung bình dự đoán lệch bao nhiêu điểm rating.

RMSE/MAE càng thấp thì mô hình dự đoán rating càng tốt.

In [ ]:
required_variables = [
    "ratings_svd",
    "ratings_train",
    "ratings_dev",
    "ratings_test",
    "movies_nlp",
    "tfidf_matrix",
    "svd_model"
]

for var_name in required_variables:
    if var_name not in globals():
        raise NameError(
            f"Bạn cần chạy các cell phía trên trước. Biến còn thiếu: {var_name}"
        )

print("Các biến cần thiết đã sẵn sàng.")
print("ratings_train:", ratings_train.shape)
print("ratings_dev:", ratings_dev.shape)
print("ratings_test:", ratings_test.shape)
print("movies_nlp:", movies_nlp.shape)
print("tfidf_matrix:", tfidf_matrix.shape)


In [ ]:
def evaluate_svd_model(model, ratings_df):
    y_true = []
    y_pred = []

    for row in ratings_df.itertuples():
        user_id = row.userId
        movie_id = row.movieId
        true_rating = row.rating

        predicted_rating = model.predict(
            user_id,
            movie_id
        ).est

        y_true.append(true_rating)
        y_pred.append(predicted_rating)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    return rmse, mae


In [ ]:
dev_rmse, dev_mae = evaluate_svd_model(
    svd_model,
    ratings_dev
)

test_rmse, test_mae = evaluate_svd_model(
    svd_model,
    ratings_test
)

svd_error_results = pd.DataFrame([
    {
        "Dataset": "Dev",
        "RMSE": dev_rmse,
        "MAE": dev_mae
    },
    {
        "Dataset": "Test",
        "RMSE": test_rmse,
        "MAE": test_mae
    }
])

svd_error_results


## 9. Xây dựng Hybrid Recommendation

Phần này xây dựng mô hình Hybrid theo đúng ý tưởng chính của đề tài:

```text
Input: userId
Output: Top N phim gợi ý cho user đó
```

Không dùng kiểu nhập tên phim rồi tìm phim tương tự, vì đó chỉ là Content-Based đơn giản.

Mô hình Hybrid gồm 2 điểm:

1. `SVDScore_norm`: điểm rating dự đoán từ SVD, chuẩn hóa về 0 đến 1.
2. `ContentScore`: điểm giống nhau giữa hồ sơ sở thích user và nội dung phim.

Công thức:

```text
HybridScore = alpha × SVDScore_norm + (1 - alpha) × ContentScore
```

### 9.1. Cấu hình chung cho Hybrid

- `RELEVANCE_THRESHOLD = 3.5`: rating từ 3.5 trở lên được xem là phim user thích.
- `K = 10`: đánh giá Top-10 phim.
- `ALPHA_LIST = [0.5, 0.7, 0.9]`: các giá trị alpha để thử trên tập dev.

In [ ]:
RELEVANCE_THRESHOLD = 3.5
K = 10
ALPHA_LIST = [0.5, 0.7, 0.9]


### 9.2. Tạo các biến tra cứu

Các biến này giúp quá trình gợi ý nhanh và rõ ràng hơn:

- `user_train_items`: lưu các phim mà mỗi user đã rating trong tập train.
- `movie_id_to_index`: ánh xạ từ `movieId` sang index trong `tfidf_matrix`.

In [ ]:
user_train_items = (
    ratings_train
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

movie_id_to_index = pd.Series(
    movies_nlp.index,
    index=movies_nlp["movieId"]
).to_dict()

print("Số user trong Train:", len(user_train_items))
print("Số phim trong movie_id_to_index:", len(movie_id_to_index))


### 9.3. Chuẩn hóa điểm SVD

SVD dự đoán rating trong khoảng 0.5 đến 5.0. Nhưng `ContentScore` nằm trong khoảng 0 đến 1.  
Vì vậy cần chuẩn hóa SVD về 0 đến 1 để có thể cộng với ContentScore.

In [ ]:
def normalize_svd_score(score, min_rating=0.5, max_rating=5.0):
    normalized_score = (score - min_rating) / (max_rating - min_rating)
    return max(0, min(1, normalized_score))


### 9.4. Xây dựng hồ sơ sở thích người dùng

Hệ thống lấy các phim user rating cao trong `ratings_train`, sau đó lấy vector TF-IDF của các phim đó.

Hồ sơ user được tính bằng trung bình có trọng số:

```text
UserProfile = trung bình các vector phim user thích, có trọng số là rating
```

Phim rating cao hơn sẽ ảnh hưởng mạnh hơn đến hồ sơ sở thích.

In [ ]:
def build_user_profile(
    user_id,
    ratings_df,
    threshold=3.5
):
    liked_ratings = ratings_df[
        (ratings_df["userId"] == user_id) &
        (ratings_df["rating"] >= threshold)
    ]

    if liked_ratings.empty:
        return None

    movie_vectors = []
    weights = []

    for row in liked_ratings.itertuples():
        movie_id = row.movieId

        if movie_id in movie_id_to_index:
            movie_index = movie_id_to_index[movie_id]
            movie_vectors.append(tfidf_matrix[movie_index])
            weights.append(row.rating)

    if len(movie_vectors) == 0:
        return None

    movie_matrix = vstack(movie_vectors)
    weights = np.array(weights)

    user_profile = (
        movie_matrix.multiply(weights[:, None]).sum(axis=0)
        / weights.sum()
    )

    return np.asarray(user_profile)


### 9.5. Hàm gợi ý phim cho một user

Hàm này là hàm chính của hệ thống.

Với một `userId`, hàm sẽ:

1. Lấy danh sách phim user đã rating trong train.
2. Loại các phim đó ra khỏi danh sách gợi ý.
3. Tính điểm SVD cho từng phim chưa xem.
4. Tính điểm nội dung nếu user có đủ phim rating cao để tạo hồ sơ.
5. Tính `hybrid_score`.
6. Sắp xếp và trả về Top N phim.

In [ ]:
def recommend_movies_for_user(
    user_id,
    alpha=0.7,
    top_n=10,
    threshold=3.5,
    method="hybrid"
):
    rated_items = user_train_items.get(user_id, set())

    candidate_movies = movies_nlp[
        ~movies_nlp["movieId"].isin(rated_items)
    ].copy()

    user_profile = build_user_profile(
        user_id=user_id,
        ratings_df=ratings_train,
        threshold=threshold
    )

    if user_profile is not None:
        content_scores = cosine_similarity(
            user_profile,
            tfidf_matrix
        ).flatten()
    else:
        content_scores = None

    recommendations = []

    for row in candidate_movies.itertuples():
        movie_id = row.movieId

        svd_pred_rating = svd_model.predict(
            user_id,
            movie_id
        ).est

        svd_score_norm = normalize_svd_score(
            svd_pred_rating
        )

        if content_scores is not None and movie_id in movie_id_to_index:
            content_score = content_scores[
                movie_id_to_index[movie_id]
            ]
        else:
            content_score = 0

        hybrid_score = (
            alpha * svd_score_norm
            + (1 - alpha) * content_score
        )

        if method == "svd":
            final_score = svd_score_norm
        elif method == "hybrid":
            final_score = hybrid_score
        else:
            raise ValueError("method phải là 'svd' hoặc 'hybrid'.")

        recommendations.append({
            "movieId": movie_id,
            "title": row.title,
            "genres": row.genres,
            "svd_pred_rating": svd_pred_rating,
            "svd_score_norm": svd_score_norm,
            "content_score": content_score,
            "hybrid_score": hybrid_score,
            "final_score": final_score
        })

    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return pd.DataFrame({
            "message": [f"Không còn phim phù hợp để gợi ý cho user {user_id}."]
        })

    return (
        recommendations_df
        .sort_values("final_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


## 10. Đánh giá Top-K

RMSE/MAE chỉ đánh giá SVD dự đoán rating lệch bao nhiêu.  
Nhưng hệ gợi ý còn cần đánh giá danh sách Top phim gợi ý.

Phần này dùng 3 chỉ số:

- `Precision@K`: trong K phim gợi ý, có bao nhiêu phim thật sự phù hợp.
- `Recall@K`: trong các phim phù hợp của user, hệ thống tìm lại được bao nhiêu phim.
- `NDCG@K`: đánh giá cả độ đúng và thứ tự xếp hạng. Phim phù hợp nằm càng cao thì NDCG càng tốt.

Tập `dev` dùng để chọn alpha tốt nhất.  
Tập `test` dùng để đánh giá cuối cùng.

### 10.1. Lấy phim phù hợp trong dev/test

Trong phần đánh giá Top-K, phim được coi là phù hợp nếu user rating từ `RELEVANCE_THRESHOLD` trở lên.

In [ ]:
def get_relevant_items(ratings_df, threshold=3.5):
    relevant_df = ratings_df[
        ratings_df["rating"] >= threshold
    ]

    relevant_items = (
        relevant_df
        .groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )

    return relevant_items


### 10.2. Tính Precision@K, Recall@K, NDCG@K

In [ ]:
def precision_recall_ndcg_at_k(
    recommended_items,
    relevant_items,
    k=10
):
    recommended_items = recommended_items[:k]

    if len(relevant_items) == 0:
        return None, None, None

    hits = [
        1 if movie_id in relevant_items else 0
        for movie_id in recommended_items
    ]

    precision = sum(hits) / k
    recall = sum(hits) / len(relevant_items)

    dcg = 0
    for i, hit in enumerate(hits):
        dcg += hit / np.log2(i + 2)

    ideal_hits = [1] * min(len(relevant_items), k)

    idcg = 0
    for i, hit in enumerate(ideal_hits):
        idcg += hit / np.log2(i + 2)

    ndcg = dcg / idcg if idcg > 0 else 0

    return precision, recall, ndcg


### 10.3. Hàm đánh giá Top-K

Hàm này dùng chung cho cả `svd` và `hybrid`, nên không cần viết nhiều hàm gợi ý trùng lặp.

In [ ]:
def evaluate_topk_model(
    ratings_eval,
    method="hybrid",
    alpha=0.7,
    k=10,
    threshold=3.5
):
    relevant_items_dict = get_relevant_items(
        ratings_eval,
        threshold=threshold
    )

    precision_list = []
    recall_list = []
    ndcg_list = []

    for user_id, relevant_items in relevant_items_dict.items():

        recommendations = recommend_movies_for_user(
            user_id=user_id,
            alpha=alpha,
            top_n=k,
            threshold=threshold,
            method=method
        )

        if "message" in recommendations.columns:
            continue

        recommended_items = recommendations["movieId"].tolist()

        precision, recall, ndcg = precision_recall_ndcg_at_k(
            recommended_items=recommended_items,
            relevant_items=relevant_items,
            k=k
        )

        if precision is not None:
            precision_list.append(precision)
            recall_list.append(recall)
            ndcg_list.append(ndcg)

    return {
        "Model": method,
        "Alpha": alpha if method == "hybrid" else "-",
        f"Precision@{k}": np.mean(precision_list),
        f"Recall@{k}": np.mean(recall_list),
        f"NDCG@{k}": np.mean(ndcg_list),
        "Number of Users": len(precision_list)
    }


### 10.4. Đánh giá trên Dev để chọn alpha

Ta thử nhiều giá trị alpha trên tập dev.  
Ở đây chọn alpha tốt nhất theo `NDCG@10`, vì hệ gợi ý không chỉ cần đúng phim mà còn cần xếp phim phù hợp lên vị trí cao.

In [ ]:
dev_results = []

dev_results.append(
    evaluate_topk_model(
        ratings_eval=ratings_dev,
        method="svd",
        k=K,
        threshold=RELEVANCE_THRESHOLD
    )
)

for alpha in ALPHA_LIST:
    dev_results.append(
        evaluate_topk_model(
            ratings_eval=ratings_dev,
            method="hybrid",
            alpha=alpha,
            k=K,
            threshold=RELEVANCE_THRESHOLD
        )
    )

dev_results_df = pd.DataFrame(dev_results)

dev_results_df


In [ ]:
hybrid_dev_results = dev_results_df[
    dev_results_df["Model"] == "hybrid"
].copy()

best_row = hybrid_dev_results.sort_values(
    by=f"NDCG@{K}",
    ascending=False
).iloc[0]

best_alpha = float(best_row["Alpha"])

print("Alpha tốt nhất theo NDCG@10 trên tập Dev:", best_alpha)


### 10.5. Đánh giá cuối cùng trên Test

Sau khi chọn được `best_alpha` trên dev, ta đánh giá trên test.  
Không dùng test để chọn alpha, vì test chỉ nên dùng cho kết quả cuối cùng.

In [ ]:
test_results = []

test_results.append(
    evaluate_topk_model(
        ratings_eval=ratings_test,
        method="svd",
        k=K,
        threshold=RELEVANCE_THRESHOLD
    )
)

test_results.append(
    evaluate_topk_model(
        ratings_eval=ratings_test,
        method="hybrid",
        alpha=best_alpha,
        k=K,
        threshold=RELEVANCE_THRESHOLD
    )
)

test_results_df = pd.DataFrame(test_results)

test_results_df


## 11. Demo gợi ý

Phần này dùng để minh họa hệ thống với một user cụ thể.

Đây là phần có thể chụp ảnh đưa vào mục demo trong báo cáo:

1. Thể loại user có xu hướng thích.
2. Top phim gợi ý cho user.
3. So sánh kết quả khi thay đổi alpha.

### 11.1. Hàm thống kê thể loại user yêu thích

Hàm này lấy các phim user rating cao trong train, sau đó thống kê thể loại xuất hiện nhiều và có rating trung bình cao.

In [ ]:
def get_user_favorite_genres(
    user_id,
    ratings_df,
    movies_df,
    min_rating=3.5
):
    user_ratings = ratings_df[
        (ratings_df["userId"] == user_id) &
        (ratings_df["rating"] >= min_rating)
    ].copy()

    if user_ratings.empty:
        return pd.DataFrame(
            columns=["genre", "count", "avg_rating"]
        )

    user_movies = user_ratings.merge(
        movies_df[["movieId", "genres"]],
        on="movieId",
        how="left"
    )

    genre_stats = {}

    for row in user_movies.itertuples():
        genres = str(row.genres).split()

        for genre in genres:
            if genre not in genre_stats:
                genre_stats[genre] = {
                    "ratings": [],
                    "count": 0
                }

            genre_stats[genre]["ratings"].append(row.rating)
            genre_stats[genre]["count"] += 1

    result = []

    for genre, stats in genre_stats.items():
        result.append({
            "genre": genre,
            "count": stats["count"],
            "avg_rating": np.mean(stats["ratings"])
        })

    return (
        pd.DataFrame(result)
        .sort_values(
            by=["avg_rating", "count"],
            ascending=False
        )
        .reset_index(drop=True)
    )


### 11.2. Demo với user cụ thể

In [ ]:
demo_user_id = 1

favorite_genres = get_user_favorite_genres(
    user_id=demo_user_id,
    ratings_df=ratings_train,
    movies_df=movies_nlp,
    min_rating=RELEVANCE_THRESHOLD
)

favorite_genres.head(10)


In [ ]:
demo_recommendations = recommend_movies_for_user(
    user_id=demo_user_id,
    alpha=best_alpha,
    top_n=10,
    threshold=RELEVANCE_THRESHOLD,
    method="hybrid"
)

demo_recommendations[
    [
        "movieId",
        "title",
        "genres",
        "svd_pred_rating",
        "svd_score_norm",
        "content_score",
        "hybrid_score"
    ]
]


### 11.3. So sánh gợi ý với nhiều alpha

Phần này giúp quan sát khi tăng/giảm alpha thì danh sách gợi ý thay đổi như thế nào.

In [ ]:
for alpha in ALPHA_LIST:
    print("=" * 80)
    print(f"Top 5 phim gợi ý cho user {demo_user_id} với alpha = {alpha}")

    result = recommend_movies_for_user(
        user_id=demo_user_id,
        alpha=alpha,
        top_n=5,
        threshold=RELEVANCE_THRESHOLD,
        method="hybrid"
    )

    display(
        result[
            [
                "title",
                "genres",
                "svd_pred_rating",
                "svd_score_norm",
                "content_score",
                "hybrid_score"
            ]
        ]
    )


## 12. Lưu kết quả và model

Phần này lưu:

1. Kết quả đánh giá SVD bằng RMSE/MAE.
2. Kết quả đánh giá Top-K trên dev.
3. Kết quả đánh giá Top-K trên test.
4. Dữ liệu phim đã xử lý.
5. Các file model `.pkl` để dùng cho Streamlit.

Các file `.pkl` dùng để app Streamlit load lại model mà không cần train lại từ đầu.

In [ ]:
svd_error_results.to_csv(
    "data/svd_error_results.csv",
    index=False
)

dev_results_df.to_csv(
    "data/evaluation_dev_results.csv",
    index=False
)

test_results_df.to_csv(
    "data/evaluation_test_results.csv",
    index=False
)

movies_nlp.to_csv(
    "data/movies_nlp_processed.csv",
    index=False
)

print("Đã lưu các file kết quả vào thư mục data.")


### 12.1. Lưu model cho Streamlit

Nếu app Streamlit nằm trong thư mục `movie_rcm_demo`, các file model sẽ được lưu vào:

```text
movie_rcm_demo/model/
```

In [ ]:
os.makedirs("movie_rcm_demo/model", exist_ok=True)

with open("movie_rcm_demo/model/svd_model.pkl", "wb") as f:
    pickle.dump(svd_model, f)

with open("movie_rcm_demo/model/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("movie_rcm_demo/model/movies_nlp.pkl", "wb") as f:
    pickle.dump(movies_nlp, f)

with open("movie_rcm_demo/model/ratings_train.pkl", "wb") as f:
    pickle.dump(ratings_train, f)

with open("movie_rcm_demo/model/movie_id_to_index.pkl", "wb") as f:
    pickle.dump(movie_id_to_index, f)

with open("movie_rcm_demo/model/best_alpha.pkl", "wb") as f:
    pickle.dump(best_alpha, f)

print("Đã lưu model vào movie_rcm_demo/model")
